# TAREA COMPLEMENTARIA - APRENDIZAJE DE REFUERZO
# AGENTE ROTO vs CORREGIDO

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

# PARTE A — AGENTE ROTO vs CORREGIDO

In [3]:
# =============================================================================
print("=" * 65)
print("PARTE A — CartPole: Agente Roto vs Corregido")
print("=" * 65)

N_BINS  = 10
limites = [(-2.4, 2.4), (-3.0, 3.0), (-0.3, 0.3), (-3.0, 3.0)]
bins    = [np.linspace(l, h, N_BINS - 1) for l, h in limites]

def disc_cp(obs):
    return tuple(int(np.digitize(obs[i], bins[i])) for i in range(4))

def moving_avg(x, w=100):
    return np.convolve(x, np.ones(w)/w, mode='valid')

def run_broken(n=3000):
    env = gym.make('CartPole-v1')
    Q = np.zeros([N_BINS]*4 + [2])
    ALPHA=0.1; GAMMA=0.0; EPSILON=1.0; EPS_MIN=0.01; EPS_DECAY=1.0
    rewards = []
    for _ in range(n):
        obs,_ = env.reset(); s=disc_cp(obs); total=0; done=False
        while not done:
            a = np.argmax(Q[s])                                         # ERROR 3
            obs2,r,done,trunc,_ = env.step(a)
            s2=disc_cp(obs2); done=done or trunc
            Q[s][a] = ALPHA*(r + GAMMA*np.max(Q[s2]) - Q[s][a])        # ERROR 4
            s=s2; total+=r
        rewards.append(total)
        EPSILON = max(EPS_MIN, EPSILON*EPS_DECAY)
    env.close(); return Q, rewards

def run_fixed(n=3000):
    env = gym.make('CartPole-v1')
    Q = np.zeros([N_BINS]*4 + [2])
    ALPHA=0.1; GAMMA=0.99; EPSILON=1.0; EPS_MIN=0.01; EPS_DECAY=0.995  # FIXES 1,2
    rewards = []
    for _ in range(n):
        obs,_ = env.reset(); s=disc_cp(obs); total=0; done=False
        while not done:
            if np.random.random() < EPSILON:                            # FIX 3
                a = env.action_space.sample()
            else:
                a = np.argmax(Q[s])
            obs2,r,done,trunc,_ = env.step(a)
            s2=disc_cp(obs2); done=done or trunc
            Q[s][a] += ALPHA*(r + GAMMA*np.max(Q[s2]) - Q[s][a])       # FIX 4
            s=s2; total+=r
        rewards.append(total)
        EPSILON = max(EPS_MIN, EPSILON*EPS_DECAY)
    env.close(); return Q, rewards

print("Entrenando agente roto...")
Q_br, rew_br = run_broken(3000)
print(f"  Últimos 100 ep: {np.mean(rew_br[-100:]):.1f}")
print("Entrenando agente corregido...")
Q_fx, rew_fx = run_fixed(3000)
print(f"  Últimos 100 ep: {np.mean(rew_fx[-100:]):.1f}")

import os # Import the os module to create directories
output_dir = '/mnt/user-data/outputs/'
os.makedirs(output_dir, exist_ok=True) # Create the directory if it doesn't exist

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(moving_avg(rew_br), color='#e74c3c', linewidth=2, label='Agente Roto (4 errores)')
ax.plot(moving_avg(rew_fx), color='#2ecc71', linewidth=2, label='Agente Corregido')
thresh = np.where(moving_avg(rew_fx) > 50)[0]
if len(thresh):
    ax.axvline(thresh[0], color='#f39c12', linestyle='--', linewidth=1.5,
               label=f'Despegue del corregido (~ep {thresh[0]})')
ax.set_xlabel('Episodio'); ax.set_ylabel('Recompensa (media móvil 100 ep)')
ax.set_title('CartPole — Agente Roto vs Agente Corregido', fontweight='bold', fontsize=13)
ax.legend(fontsize=10); ax.grid(alpha=0.3); ax.set_ylim(0)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/t3_roto_vs_corregido.png', bbox_inches='tight')
plt.close()
print("Gráfico Parte A guardado.")

def evaluate(Q, n=20):
    env = gym.make('CartPole-v1'); rews = []
    for _ in range(n):
        obs,_ = env.reset(); s=disc_cp(obs); total=0; done=False
        while not done:
            a=np.argmax(Q[s]); obs2,r,done,trunc,_=env.step(a)
            s=disc_cp(obs2); done=done or trunc; total+=r
        rews.append(total)
    env.close(); return rews

ev_br = evaluate(Q_br); ev_fx = evaluate(Q_fx)
print(f"\nEvaluación (20 ep sin exploración):")
print(f"  Roto     → Media:{np.mean(ev_br):.1f}  Max:{np.max(ev_br):.0f}  Min:{np.min(ev_br):.0f}")
print(f"  Corregido→ Media:{np.mean(ev_fx):.1f}  Max:{np.max(ev_fx):.0f}  Min:{np.min(ev_fx):.0f}")

PARTE A — CartPole: Agente Roto vs Corregido
Entrenando agente roto...
  Últimos 100 ep: 9.4
Entrenando agente corregido...
  Últimos 100 ep: 50.4
Gráfico Parte A guardado.

Evaluación (20 ep sin exploración):
  Roto     → Media:9.3  Max:11  Min:8
  Corregido→ Media:49.8  Max:109  Min:12


# PARTE B — MountainCar

In [4]:
# =============================================================================
print("\n" + "=" * 65)
print("PARTE B — MountainCar-v0")
print("=" * 65)

env_mc = gym.make('MountainCar-v0')
print(f"Obs space: {env_mc.observation_space}  |  Action space: {env_mc.action_space}")
rnd_rews = []
for i in range(5):
    obs,_ = env_mc.reset(); total=0; done=False
    while not done:
        obs,r,done,trunc,_ = env_mc.step(env_mc.action_space.sample())
        done=done or trunc; total+=r
    rnd_rews.append(total)
    print(f"  Episodio aleatorio {i+1}: {total:.0f}")
env_mc.close()
print(f"  Promedio aleatorio: {np.mean(rnd_rews):.1f}")

MC_BINS = 20
mc_bins = [
    np.linspace(-1.2, 0.6,  MC_BINS-1),
    np.linspace(-0.07, 0.07, MC_BINS-1),
]
def disc_mc(obs):
    return tuple(int(np.digitize(obs[i], mc_bins[i])) for i in range(2))

def train_mc(n=10000):
    env = gym.make('MountainCar-v0')
    Q = np.zeros([MC_BINS, MC_BINS, 3])
    ALPHA=0.2; GAMMA=0.99; EPSILON=1.0; EPS_MIN=0.01; EPS_DECAY=0.9995
    rewards = []; successes = 0
    for ep in range(n):
        obs,_ = env.reset(); s=disc_mc(obs); total=0; done=False
        while not done:
            if np.random.random() < EPSILON:
                a = env.action_space.sample()
            else:
                a = np.argmax(Q[s])
            obs2,r,done,trunc,_ = env.step(a)
            s2 = disc_mc(obs2); done = done or trunc
            # Reward shaping: incentivo por ganar altura y velocidad positiva
            pos, vel = obs2
            r_s = r + 10*(pos - (-0.5)) + 5*abs(vel)
            Q[s][a] += ALPHA*(r_s + GAMMA*np.max(Q[s2]) - Q[s][a])
            s=s2; total+=r
        if total > -200: successes += 1
        rewards.append(total)
        EPSILON = max(EPS_MIN, EPSILON*EPS_DECAY)
    env.close()
    print(f"  Episodios exitosos: {successes}/{n} ({successes/n*100:.1f}%)")
    return Q, rewards

print("\nEntrenando MountainCar (10000 ep)...")
Q_mc, rew_mc = train_mc(10000)
print(f"  Recompensa media últimos 200 ep: {np.mean(rew_mc[-200:]):.1f}")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(np.arange(len(rew_mc)), rew_mc, color='#3498db', alpha=0.12, linewidth=0.5)
ma_mc = moving_avg(rew_mc, 300)
ax.plot(np.arange(299, len(rew_mc)), ma_mc, color='#1a5276', linewidth=2.5,
        label='Media móvil 300 ep')
ax.axhline(-200, color='#e74c3c', linestyle='--', linewidth=1.5, label='Agente aleatorio')
ax.set_xlabel('Episodio'); ax.set_ylabel('Recompensa total')
ax.set_title('MountainCar-v0 — Curva de Aprendizaje Q-Learning', fontweight='bold', fontsize=13)
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/t3_mountaincar_curva.png', bbox_inches='tight')
plt.close()
print(f"Q-table MountainCar: {Q_mc.shape} = {Q_mc.size} entradas")


PARTE B — MountainCar-v0
Obs space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)  |  Action space: Discrete(3)
  Episodio aleatorio 1: -200
  Episodio aleatorio 2: -200
  Episodio aleatorio 3: -200
  Episodio aleatorio 4: -200
  Episodio aleatorio 5: -200
  Promedio aleatorio: -200.0

Entrenando MountainCar (10000 ep)...
  Episodios exitosos: 0/10000 (0.0%)
  Recompensa media últimos 200 ep: -200.0
Q-table MountainCar: (20, 20, 3) = 1200 entradas


# PARTE C — ENTORNO PROPIO: AGENTE DE INVENTARIO

In [5]:
# =============================================================================
print("\n" + "=" * 65)
print("PARTE C — Agente de Inventario")
print("=" * 65)

class InventarioEnv:
    """
    MDP — Gestión de inventario diaria (30 días por episodio)
    S: (bin_stock [0-9], bin_demanda [0-3])
    A: {0:no pedir, 1:pedir 5, 2:pedir 10, 3:pedir 20}
    R: ventas*10 - stock*0.5 - quiebre*3
    γ: 0.95
    """
    ORDERS = [0, 5, 10, 20]
    MAX_STOCK = 50; MAX_DIAS = 30

    def __init__(self):
        self.stock_bins  = np.linspace(0, self.MAX_STOCK, 11)[1:-1]  # 9 cortes → 10 bins
        self.dem_bins    = np.array([2, 5, 8])                        # 3 cortes → 4 bins

    def _s(self):
        sb = int(np.digitize(self.stock,  self.stock_bins))   # 0-9
        db = int(np.digitize(self.demanda, self.dem_bins))    # 0-3
        return (sb, db)

    def reset(self):
        self.stock   = int(np.random.randint(5, 25))
        self.demanda = int(np.random.randint(0, 7))
        self.dia = 0
        return self._s(), {}

    def step(self, action):
        order = self.ORDERS[action]
        self.stock = min(self.stock + order, self.MAX_STOCK)
        dem = int(np.clip(np.random.poisson(4), 0, 12))
        ventas  = min(dem, self.stock)
        quiebre = max(0, dem - self.stock)
        self.stock  = max(0, self.stock - dem)
        self.demanda = dem
        reward = ventas*10 - self.stock*0.5 - quiebre*3
        self.dia += 1
        return self._s(), reward, self.dia >= self.MAX_DIAS

def train_inv(n=6000):
    env = InventarioEnv()
    Q = np.zeros([10, 4, 4])   # 10 bins stock × 4 bins dem × 4 acciones
    ALPHA=0.15; GAMMA=0.95; EPSILON=1.0; EPS_MIN=0.02; EPS_DECAY=0.997
    rewards = []
    for _ in range(n):
        s,_ = env.reset(); total=0; done=False
        while not done:
            if np.random.random() < EPSILON:
                a = np.random.randint(4)
            else:
                a = np.argmax(Q[s])
            s2,r,done = env.step(a)
            Q[s][a] += ALPHA*(r + GAMMA*np.max(Q[s2]) - Q[s][a])
            s=s2; total+=r
        rewards.append(total)
        EPSILON = max(EPS_MIN, EPSILON*EPS_DECAY)
    return Q, rewards

print("Entrenando agente de inventario (6000 ep)...")
Q_inv, rew_inv = train_inv(6000)
print(f"  Primeros 200 ep: {np.mean(rew_inv[:200]):.1f}")
print(f"  Últimos  200 ep: {np.mean(rew_inv[-200:]):.1f}")
mejora = np.mean(rew_inv[-200:]) - np.mean(rew_inv[:200])
print(f"  Mejora: {mejora:+.1f}")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(np.arange(len(rew_inv)), rew_inv, color='#9b59b6', alpha=0.12, linewidth=0.5)
ma_inv = moving_avg(rew_inv, 100)
ax.plot(np.arange(99, len(rew_inv)), ma_inv, color='#6c3483', linewidth=2.5, label='Media móvil 100 ep')
ax.set_xlabel('Episodio (30 días cada uno)'); ax.set_ylabel('Recompensa total')
ax.set_title('Agente de Inventario — Curva de Aprendizaje Q-Learning',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/t3_inventario_curva.png', bbox_inches='tight')
plt.close()

# Demo 1 episodio
print("\n--- DEMO 1 episodio (agente entrenado) ---")
env_d = InventarioEnv(); s,_ = env_d.reset()
total_d=0; done=False; dia=0
names = ['No pedir','Pedir 5','Pedir 10','Pedir 20']
print(f"{'Día':>4}|{'Stock':>6}|{'Acción':>10}|{'Reward':>8}|{'Acum':>8}")
print("-"*42)
while not done:
    a = np.argmax(Q_inv[s]); s2,r,done=env_d.step(a)
    total_d+=r; dia+=1; s=s2
    if dia<=8:
        print(f"{dia:>4}|{env_d.stock:>6}|{names[a]:>10}|{r:>8.1f}|{total_d:>8.1f}")
print(f"  ... total 30 días: {total_d:.1f}")

# Resumen 3 partes
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Aprendizaje por Refuerzo — Resumen Partes A · B · C',
             fontsize=13, fontweight='bold')
axes[0].plot(moving_avg(rew_br), '#e74c3c', linewidth=2, label='Roto')
axes[0].plot(moving_avg(rew_fx), '#2ecc71', linewidth=2, label='Corregido')
axes[0].set_title('A: CartPole'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Episodio'); axes[0].set_ylabel('Recompensa (MA100)')
axes[1].plot(moving_avg(rew_mc, 300), '#1a5276', linewidth=2.5)
axes[1].axhline(-200, color='#e74c3c', linestyle='--')
axes[1].set_title('B: MountainCar'); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Episodio'); axes[1].set_ylabel('Recompensa (MA300)')
axes[2].plot(moving_avg(rew_inv, 100), '#6c3483', linewidth=2.5)
axes[2].set_title('C: Inventario'); axes[2].grid(alpha=0.3)
axes[2].set_xlabel('Episodio'); axes[2].set_ylabel('Recompensa (MA100)')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/t3_resumen_3partes.png', bbox_inches='tight')
plt.close()


PARTE C — Agente de Inventario
Entrenando agente de inventario (6000 ep)...
  Primeros 200 ep: 578.6
  Últimos  200 ep: 830.3
  Mejora: +251.7

--- DEMO 1 episodio (agente entrenado) ---
 Día| Stock|    Acción|  Reward|    Acum
------------------------------------------
   1|    17|  No pedir|    41.5|    41.5
   2|    13|  No pedir|    33.5|    75.0
   3|    19|  Pedir 10|    30.5|   105.5
   4|    18|  No pedir|     1.0|   106.5
   5|    19|   Pedir 5|    30.5|   137.0
   6|    15|  No pedir|    32.5|   169.5
   7|    13|  No pedir|    13.5|   183.0
   8|    19|  Pedir 10|    30.5|   213.5
  ... total 30 días: 832.5
